In [26]:
import numpy as np
import pandas as pd
np.random.seed(100)

Here I'm going to make some fake data. The idea is i want long data with irregular doctor time visits and some number of features for each patient that's recorded in each visit. We are going to try and predict using the graphs "was this patient sick or not".

I'll have, $N$ many subjects (say 500), and we'll have $n_i$ be the number of visits per subject. The number of times they visit is then going to be $n_i\sim U(3,10)$

we'll have 2 underlying variables per patient, one being, z, the "how many times they get sick" and another, v, being "how intense is their sickness". these can be simply random.
$$z_i\sim Poisson(1)\\ v_i\sim N(0,1)$$
and we'll have lower v means less intense symptoms.

The "sick days" are determined randomly. so if $z_i > 0$, then we would randomly select (uniformly) which visits were "sick visits", just a binary counter.

what we'll set up is that, for each patient, each time they get sick their random features will change by some factor of $v_i$. For some patients, if they have $v=2$ or something, they will see a big shift in features which will be tell-tale that they got sick. 

Also, the times between visits, or "gaps" will also be determined by both $z$ and $v$. if they "get sick" they will come to the doctor 'soon' after their last visit, this can be determined with something like this, $$t_{ij}\sim exponential\left(\frac{exp(-v_i)}{1+exp(-v_i)}*I(\text{is sick}) + 1\right)$$

we can have $p=5$ features, $x_1,\dots,x_5$. and we'll have $x_1$ not influence the outcome, but the rest do. They can be determined like this:
$$
x_{ij1} = 5 + \frac{exp(v_i)}{1+exp(v_i)}*I(\text{is sick}) + \epsilon_ij
x_{ij1} = 5 + \frac{exp(v_i)}{1+exp(v_i)}*I(\text{is sick}) + \epsilon_ij
x_{ij1} = 5 + \frac{exp(v_i)}{1+exp(v_i)}*I(\text{is sick}) + \epsilon_ij
x_{ij1} = 5 + \frac{exp(v_i)}{1+exp(v_i)}*I(\text{is sick}) + \epsilon_ij
x_{ij1} = 5 + \frac{exp(v_i)}{1+exp(v_i)}*I(\text{is sick}) + \epsilon_ij
$$
something like that (please change these to be different)

We'll have the outcome literally just be some probaility that they got sick or not. It should be roughly 0.368 that don't get sick. or around 36.8% of people. We'll of course have the number. 


In [ ]:
# num patients
N = 500
# num features
p = 5

# controls the number of visits per patient
z_i = np.random.normal(loc=6, scale=1, size=N)

n_i = np.random.poisson(lam=z_i).astype(int)

z_i[:4], n_i[:4]

(array([6.38059753, 4.46076016, 7.20695005, 6.13861587]), array([4, 4, 5, 9]))

In [ ]:

# create a list of patient ids
patients = []
for i in range(N):
    for j in range(n_i[i]):
        patients.append({
            "patient_id": i + 1 ,
            "visit_id": j + 1
        })
data = pd.DataFrame(patients)

In [30]:
N, p, n_i[:4], data.head(n=15)

(500,
 5,
 array([ 3,  3,  6, 10], dtype=int32),
     patient_id  visit_id
 0            1         1
 1            1         2
 2            1         3
 3            2         1
 4            2         2
 5            2         3
 6            3         1
 7            3         2
 8            3         3
 9            3         4
 10           3         5
 11           3         6
 12           4         1
 13           4         2
 14           4         3)

Now I have some list of patients with visits, and the visits are between 3 and 10 times. 

Next I want to make visit times. These can be exponential because I need something continuous, monotone, and unbounded. 

Have $t_ij\sim Exponential(\lambda)$ be the visit time of patient $i$ and visit $j$. Here $\lambda$ is mostly unimportant, so we'll do $\lambda=2$

In [ ]:
visit_times = []

for i in range(N):
    gaps = np.random.exponential(scale = 1, size = n_i[i] - 1) # the -1 here skips the first visit, since it has to be at time = 0.
    times = np.concatenate(([0], np.cumsum(gaps))) # the first visit is at time = 0, so we concatenate that to the cumulative sum of the gaps.
    visit_times.extend(times)

data["time"] = visit_times

In [37]:
data[:15]

,patient_id,visit_id,time
0,1,1,0.000000
1,1,2,0.369468
2,1,3,3.006198
3,2,1,0.000000
4,2,2,0.267850
5,2,3,2.589301
6,3,1,0.000000
7,3,2,1.857313
8,3,3,2.846043
9,3,4,4.798492


The idea now is the make some features. I want 5, as stated before, and we'll have some latent driver for each patient, $z_i\sim Normal(0,1)$. 

In this, we'll have the relationship that
$$
x_{ij1} \sim N(2*z_{i} + 0.2t_{ij}, 1)\\
x_{ij2} \sim N(0.5t_{ij}, 1)\\
x_{ij3} \sim N(0.8*z_{i} + 0.2t_{ij}, 1)\\
x_{ij4} \sim N(0.5*z_{i} - 0.3t_{ij}, 1)\\
x_{ij5} \sim N(0.3*z_{i} + 0.2t_{ij}, 1)\\
$$